# Model refit — fit the Section 4 finalists on all development data, score Test

**Depends on:** Section 1 (data split), Section 3 (saved experiment configs),
Section 4 (`finalist_choice.parquet`).
**Produces:** `../Evaluation pipeline/outputs/test_predictions.parquet` for
Layer A/B/C evaluation in the Evaluation pipeline.

Section 3's experiments fit per fold and never save a production model.
This notebook does the single "fit on all eligible development rows,
then score Test" step the deployment story needs.

The two finalists frozen by Section 4:

| Hotel | Finalist | Components | Calibration |
|---|---|---|---|
| H1 | H1-E2 | RF30 + XGB30 (equal average) | Uncalibrated |
| H2 | H2-E1 | C2 + TabM-E8 (equal average) | Uncalibrated |

Where:
- **RF30 / XGB30** = base configs from `development_config.json`.
- **C2** = RF30 + 180-day recency weighting + FE4 (historical segment
  cancellation rate).
- **TabM-E8** = TabM with Cylvie's screening architecture, trained for 8 epochs.

The training cohort is Dev bookings with `FinalRefit == 1` — the saved
membership flag that encodes the three-way eligibility rule
(CutoffDate / LabelAvailableDate / ExpectedCheckoutDate strictly before
`test_start = 2017-04-18`). This leaves 15,998 H1 bookings and 29,431 H2
bookings for training.

**Reuses Cylvie's screening/experiment code so no training logic is
reimplemented.** `_engineer` adds FE4, `_fit_predict_sklearn` fits the
RF/XGB pipelines with sample weights for recency, and `_fit_predict_tabm`
runs the exact TabM training loop.


In [1]:
from __future__ import annotations
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd

sys.path.insert(0, str(Path.cwd()))
from utils import model_refit

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

GOLD_ROOT    = Path("../Data pipeline/hotel_booking_pipeline/datamart/gold").resolve()
MEMBERSHIP   = Path("splits/split_v2/membership.parquet").resolve()
SPLIT_CFG    = Path("splits/split_v2/config.json").resolve()
FEATURE_CT   = GOLD_ROOT / "feature_contract.json"

OUT_PATH     = Path("../Evaluation pipeline/outputs/test_predictions.parquet").resolve()
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

for p in (GOLD_ROOT, MEMBERSHIP, SPLIT_CFG, FEATURE_CT):
    if not p.exists():
        raise FileNotFoundError(f"Missing input: {p}")
print("All input paths resolve:", OUT_PATH.parent, "ok")


All input paths resolve: /workspace/Project/Evaluation pipeline/outputs ok


## 1. Load cohorts

For each hotel, build:
- `train` — Dev bookings with `FinalRefit == 1` (eligible under the three-way
  date rule, `test_start = 2017-04-18`).
- `test`  — all `Partition == "Test"` bookings.


In [2]:
cohorts = {}
for hotel in ("H1", "H2"):
    c = model_refit.load_cohorts(
        hotel=hotel,
        gold_dataset_root=GOLD_ROOT / "model_dataset",
        membership_path=MEMBERSHIP,
        split_config_path=SPLIT_CFG,
        feature_contract_path=FEATURE_CT,
    )
    cohorts[hotel] = c
    train_prev = c.train["IsCanceled"].mean()
    print(f"  [{hotel}] train n={len(c.train):>7,} (prev={train_prev:.3f})  "
          f"test n={len(c.test):>7,}  fit_date={c.fit_date.date()}")


  [H1] train n= 15,998 (prev=0.077)  test n=  3,576  fit_date=2017-04-18
  [H2] train n= 29,431 (prev=0.131)  test n=  7,748  fit_date=2017-04-18


## 2. Refit H1 — H1-E2 ensemble (RF30 + XGB30)

Each component is a fresh fit on the full H1 training cohort using the
exact estimator parameters from `H1_RF_v2/config.json` and
`H1_XGB_v2/config.json`. RF uses `model_preprocessing.build_preprocessor`
in RF mode (unscaled numeric, one-hot categorical). XGB uses the same
preprocessor in XGB mode (same shape, no scaling).

Expected runtime on CPU: RF ~1-2 min, XGB ~1 min.


In [3]:
t0 = time.perf_counter()
h1_pred, h1_detail = model_refit.refit_and_score_finalist(
    hotel="H1", cohorts=cohorts["H1"], include_tabm=True,
)
print(f"\nH1 ensemble done in {time.perf_counter() - t0:.1f}s")
print(f"Components fitted: {h1_detail['components_fitted']}")
h1_pred.head()


  [H1] fitting RF30 (RF)...
    ok  n_test=3,576  took 3.8s
  [H1] fitting XGB30 (XGB)...
    ok  n_test=3,576  took 0.6s

H1 ensemble done in 4.5s
Components fitted: ['RF30', 'XGB30']


,Hotel,Booking_ID,ArrivalDate,ADR,TotalNights,p_cancel,y_true,CutoffDate,p_RF30,p_XGB30
0,H1,H1-011304,2017-05-02,133.5000,4,0.1501,1,2017-04-18,0.1382,0.1621
1,H1,H1-011310,2017-05-02,73.5000,7,0.0511,1,2017-04-18,0.0554,0.0467
2,H1,H1-011311,2017-05-02,82.0000,9,0.0350,1,2017-04-18,0.0464,0.0237
3,H1,H1-011342,2017-05-03,54.0000,7,0.0603,1,2017-04-19,0.0655,0.0550
4,H1,H1-011351,2017-05-04,50.0000,2,0.1083,1,2017-04-20,0.0934,0.1231


## 3. Refit H2 — H2-E1 ensemble (C2 + TabM-E8)

- **C2** = RF30 + recency weighting (half-life 180 days from `test_start`)
  + FE4 (historical-segment-cancellation-rate feature engineered on the
  train split and applied to Test with the final mapping, matching
  `_engineer`'s screening-time behaviour).
- **TabM-E8** = TabM with architecture `{d_block:64, n_blocks:2, k:8,
  dropout:0.1}` trained for 8 epochs via `_fit_predict_tabm`.

Expected runtime on CPU: C2 ~2-3 min, TabM ~10-20 min (depending on
`torch.set_num_threads` and CPU). If you need to prototype only C2 for
now, pass `include_tabm=False` and the ensemble will be C2 alone — mark
the predictions file as partial.


In [4]:
t0 = time.perf_counter()
h2_pred, h2_detail = model_refit.refit_and_score_finalist(
    hotel="H2", cohorts=cohorts["H2"], include_tabm=True,
)
print(f"\nH2 ensemble done in {time.perf_counter() - t0:.1f}s")
print(f"Components fitted: {h2_detail['components_fitted']}")
if h2_detail['components_skipped']:
    print(f"Components SKIPPED: {h2_detail['components_skipped']}  "
          f"(result is a partial ensemble)")
h2_pred.head()


  [H2] fitting C2 (RF)...
    ok  n_test=7,748  took 6.9s
  [H2] fitting TabM-E8 (TabM)...
    ok  n_test=7,748  took 6.2s

H2 ensemble done in 21.4s
Components fitted: ['C2', 'TabM-E8']


,Hotel,Booking_ID,ArrivalDate,ADR,TotalNights,p_cancel,y_true,CutoffDate,p_C2,p_TabM-E8
0,H2,H2-027283,2017-05-02,112.5000,1,0.0822,1,2017-04-18,0.0904,0.0740
1,H2,H2-027287,2017-05-02,108.0000,2,0.1897,1,2017-04-18,0.1872,0.1923
2,H2,H2-027292,2017-05-02,140.0000,3,0.0852,1,2017-04-18,0.1101,0.0603
3,H2,H2-027293,2017-05-02,108.0000,3,0.1832,1,2017-04-18,0.1768,0.1895
4,H2,H2-027295,2017-05-02,120.0000,3,0.2264,1,2017-04-18,0.2026,0.2503


## 4. Combine and save Test predictions


In [5]:
combined = pd.concat([h1_pred, h2_pred], ignore_index=True)
assert combined["p_cancel"].notna().all()
assert combined["y_true"].isin([0, 1]).all()
assert (combined["p_cancel"].between(0, 1)).all()

for h in ("H1", "H2"):
    sub = combined[combined.Hotel == h]
    print(f"  {h}  n={len(sub):,}  prev={sub.y_true.mean():.3f}  mean(p)={sub.p_cancel.mean():.3f}")

combined.to_parquet(OUT_PATH, index=False)
print(f"\nSaved -> {OUT_PATH}")


  H1  n=3,576  prev=0.070  mean(p)=0.054
  H2  n=7,748  prev=0.079  mean(p)=0.110

Saved -> /workspace/Project/Evaluation pipeline/outputs/test_predictions.parquet


## 5. Persist fitted models for the deployment handoff

Each component's fitted sklearn pipeline is pickled, the C2 FE4 feature
mapping is JSON-dumped, and the TabM artifact records the exact spec +
preprocessing recipe so the deploy team can refit identically. A
``metadata.json`` manifest per hotel lists everything, SHA-256
fingerprints each artifact, captures library versions, and includes
scoring instructions.


In [6]:
MODELS_ROOT = Path("models").resolve()
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

h1_persist = model_refit.persist_hotel_models(
    hotel="H1", cohorts=cohorts["H1"], summary=h1_detail,
    models_root=MODELS_ROOT,
)
h2_persist = model_refit.persist_hotel_models(
    hotel="H2", cohorts=cohorts["H2"], summary=h2_detail,
    models_root=MODELS_ROOT,
)

print(f"\nModel registry:")
print(f"  H1 -> {h1_persist['model_version']}  at  {h1_persist['models_dir']}")
print(f"  H2 -> {h2_persist['model_version']}  at  {h2_persist['models_dir']}")


  [H1] saved 2 components -> /workspace/Project/Model pipeline/models/H1-E2_v1_20170418
  [H2] saved 2 components -> /workspace/Project/Model pipeline/models/H2-E1_v1_20170418

Model registry:
  H1 -> H1-E2_v1_20170418  at  /workspace/Project/Model pipeline/models/H1-E2_v1_20170418
  H2 -> H2-E1_v1_20170418  at  /workspace/Project/Model pipeline/models/H2-E1_v1_20170418


## 6. Handoff summary

**Files for the Evaluation pipeline (consumed by `../Evaluation pipeline/evaluation_notebook.ipynb`):**
- `../Evaluation pipeline/outputs/test_predictions.parquet` — one row
  per Test booking.

**Files for the deployment / MLOps team (new, under `models/`):**
- `models/<model_version>/metadata.json` — the manifest. Lists every
  component, its artifact file, SHA-256, feature columns, library
  versions, split fingerprint, and scoring instructions.
- `models/<model_version>/<component>_pipeline.pkl` — fitted sklearn
  pipeline for RF30, XGB30, and C2.
- `models/<model_version>/<component>_fe4_mapping.json` — C2's
  MarketSegment → historical cancel rate mapping. Apply to new bookings
  before passing them to C2's pipeline.
- `models/<model_version>/<component>_tabm_artifact.json` — TabM spec +
  preprocessing recipe. See `utils/model_scoring.py` for the scoring
  interface and the note on TabM refit (Section 3's TabM fit function
  does not expose the fitted torch module, so the deploy team refits
  from this spec; the recipe is byte-for-byte reproducible given the
  same ``FinalRefit==1`` cohort and seed).

See `utils/model_scoring.py` for the end-to-end loading and scoring
interface the deploy team should use.
